In [0]:
# 
# Slowly Changing Dimension Type 2: instead of overwriting, we historize changes.
# Each row has valid_from, valid_to, is_current — full history preserved.
# 

import sys
for key in list(sys.modules.keys()):
    if 'market_pulse' in key:
        del sys.modules[key]

sys.path.insert(0, "/Workspace/Repos/martalimas@gmail.com/market-pulse-pipeline/src")

from market_pulse.config import SILVER_DIM_STOCK_PATH, STORAGE_ACCOUNT

SCD2_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/dim_stock_scd2/"

print("✅ Configuration loaded")
print(f"  Source: {SILVER_DIM_STOCK_PATH}")
print(f"  Target: {SCD2_PATH}")

In [0]:
# ─── 1. Lê a dim_stock actual ─────────────────────────────────────────────────
df_dim = spark.read.format("delta").load(SILVER_DIM_STOCK_PATH)
df_dim.display()

In [0]:
# ─── 2. Enriquece com sector e exchange (lookup hardcoded) ────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Dados de referência — no mundo real viriam de um feed externo
stock_metadata = {
    "AAPL":    ("Technology", "NASDAQ"),
    "MSFT":    ("Technology", "NASDAQ"),
    "IBM":     ("Technology", "NYSE"),
    "EDP.LS":  ("Utilities",  "Euronext Lisbon"),
    "GALP.LS": ("Energy",     "Euronext Lisbon"),
    "BCP.LS":  ("Banking",    "Euronext Lisbon"),
}

# Cria DataFrame de referência
ref_data = [(k, v[0], v[1]) for k, v in stock_metadata.items()]
df_ref = spark.createDataFrame(ref_data, ["symbol", "sector", "exchange"])

# JOIN para enriquecer
df_enriched = df_dim.join(df_ref, on="symbol", how="left")

In [0]:
# ─── 3. Adiciona colunas SCD Type 2 ──────────────────────────────────────────
df_scd2 = (df_enriched
    .withColumn("valid_from", F.col("last_refreshed"))
    .withColumn("valid_to", F.lit("9999-12-31").cast("date"))
    .withColumn("is_current", F.lit(True))
)

df_scd2.display()

In [0]:
# ─── 4. Escreve a tabela SCD2 (seed inicial) ─────────────────────────────────
df_scd2.write.format("delta").mode("overwrite").save(SCD2_PATH)
print(f"✅ dim_stock_scd2 criada com {df_scd2.count()} rows")

In [0]:
# ─── 3. MERGE — SCD Type 2 logic ─────────────────────────────────────────────
from delta.tables import DeltaTable

def merge_scd2(spark, new_data_df, scd2_path):
    """
    Applies SCD Type 2 merge to dim_stock_scd2.
    
    Compares incoming data with existing current rows.
    If attributes changed: closes old row + inserts new row.
    If new stock: inserts.
    If nothing changed: skips.
    """
    # Columns that we track for changes
    tracked_cols = ["time_zone", "sector", "exchange"]
    
    # Build the change detection condition
    # "something changed" = any tracked column differs between old and new
    change_condition = " OR ".join(
        [f"existing.{c} <> incoming.{c}" for c in tracked_cols]
    )
    
    # Load existing SCD2 table
    scd2_table = DeltaTable.forPath(spark, scd2_path)
    
    # The MERGE
    (scd2_table.alias("existing")
        .merge(
            new_data_df.alias("incoming"),
            # Match: same stock AND current row
            "existing.symbol = incoming.symbol AND existing.is_current = true"
        )
        # WHEN MATCHED and something changed → close the old row
        .whenMatchedUpdate(
            condition=change_condition,
            set={
                "valid_to": "incoming.valid_from",
                "is_current": "false"
            }
        )
        # WHEN NOT MATCHED → new stock, insert
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    # Now insert the NEW version of changed rows
    # (the MERGE above only CLOSED the old ones — we still need to INSERT the new)
    new_rows = (new_data_df.alias("incoming")
        .join(
            spark.read.format("delta").load(scd2_path)
                .filter("is_current = false")
                .alias("closed"),
            on="symbol",
            how="inner"
        )
        .select("incoming.*")
        .distinct()
    )
    
    if new_rows.count() > 0:
        new_rows.write.format("delta").mode("append").save(scd2_path)
        print(f"  ✅ {new_rows.count()} new version(s) inserted")
    
    print("✅ SCD Type 2 merge complete")

print("✅ merge_scd2() defined")

In [0]:
# ─── 4. Simular mudança + executar MERGE ──────────────────────────────────────

# Dados "novos" — como se o pipeline corresse hoje com uma mudança
# GALP.LS foi reclassificada de Energy → Renewable Energy
from pyspark.sql import functions as F
from datetime import date

new_data = [
    ("AAPL",    "2026-05-28", "US/Eastern",  "Technology",       "NASDAQ",           date(2026,5,28), date(9999,12,31), True),
    ("MSFT",    "2026-05-28", "US/Eastern",  "Technology",       "NASDAQ",           date(2026,5,28), date(9999,12,31), True),
    ("IBM",     "2026-05-28", "US/Eastern",  "Technology",       "NYSE",             date(2026,5,28), date(9999,12,31), True),
    ("EDP.LS",  "2026-05-28", "US/Eastern",  "Utilities",        "Euronext Lisbon",  date(2026,5,28), date(9999,12,31), True),
    ("GALP.LS", "2026-05-28", "US/Eastern",  "Renewable Energy", "Euronext Lisbon",  date(2026,5,28), date(9999,12,31), True),  # ← MUDOU
    ("BCP.LS",  "2026-05-28", "US/Eastern",  "Banking",          "Euronext Lisbon",  date(2026,5,28), date(9999,12,31), True),
]

columns = ["symbol", "last_refreshed", "time_zone", "sector", "exchange", 
           "valid_from", "valid_to", "is_current"]

df_new = spark.createDataFrame(new_data, columns) \
    .withColumn("last_refreshed", F.col("last_refreshed").cast("date")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

print("📥 Incoming data (GALP.LS sector changed to Renewable Energy):")
df_new.select("symbol", "sector", "exchange").display()

# Executar o MERGE
merge_scd2(spark, df_new, SCD2_PATH)

# Ver o resultado
print("\n📊 dim_stock_scd2 after MERGE:")
spark.read.format("delta").load(SCD2_PATH) \
    .orderBy("symbol", "valid_from") \
    .display()

In [0]:
# # ─── Reset: limpa e recria o seed ─────────────────────────────────────────────
# dbutils.fs.rm(SCD2_PATH, recurse=True)
# df_scd2.write.format("delta").mode("overwrite").save(SCD2_PATH)
# print(f"✅ dim_stock_scd2 recriada com {df_scd2.count()} rows (seed limpo)")

In [0]:
# ─── Stage 5 — Delta Lake Time Travel ────────────────────────────────────────
# Every write to a Delta table creates a new version.
# Time Travel lets you query any previous version — like git for data.
# ─────────────────────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, "/Workspace/Repos/martalimas@gmail.com/market-pulse-pipeline/src")
from market_pulse.config import STORAGE_ACCOUNT

SCD2_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/dim_stock_scd2/"
FACT_PRICES_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/fact_prices/"

# ─── 1. Ver o histórico de versões ────────────────────────────────────────────
print("📜 Histórico de versões da dim_stock_scd2:")
spark.sql(f"DESCRIBE HISTORY delta.`{SCD2_PATH}`").select(
    "version", "timestamp", "operation", "operationMetrics"
).display()

In [0]:
# ─── 2. Query a uma versão anterior ──────────────────────────────────────────
# Versão 1 = o seed limpo, ANTES do MERGE (6 stocks, todos is_current=true)
print("⏪ Versão 1 — ANTES do MERGE (seed limpo):")
spark.read.format("delta").option("versionAsOf", 1).load(SCD2_PATH) \
    .orderBy("symbol") \
    .display()

# Versão actual = DEPOIS do MERGE (7 rows, GALP.LS com histórico)
print("\n⏩ Versão actual — DEPOIS do MERGE:")
spark.read.format("delta").load(SCD2_PATH) \
    .orderBy("symbol", "valid_from") \
    .display()
